# ACDC experiment: reconstruction, verification and dynamic shimming

Run this notebook **from top to bottom**. All experiment parameters are in the next cells;
no configuration JSON is read. Each stage displays its input, result and checks.

Python reconstructs the Twix data, reads the NPZ calibration and draws diagnostics.
**The forward model and optimization execute the existing MATLAB functions.**
The final cells export one waveform CSV after the numerical and waveform-data checks pass.

The example input folder is `acquisition_001`: a mono-echo 2D GRE/FLASH reference Twix,
a multi-echo B0 Twix, a 32-channel ACDC volume and an EPI scan. A conventional
mono-echo GRE scan is also supported. Each acquisition has its own Pulseq `.seq` file.
The reference, B0 and scan may use different matrix sizes and FOVs when their physical
grids cover the selected ROI. The ROI is defined from the reconstructed reference
in step 4b and displayed before checking map coverage.

Figures stay in the notebook. The output folder receives only `waveform.csv`; no FPGA control is performed.


In [ ]:
# 1. Imports and paths
%matplotlib inline
import hashlib
import os
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "Experiment").is_dir() else Path.cwd().parent
EXPERIMENT_DIR = PROJECT_ROOT / "Experiment"
sys.path.insert(0, str(EXPERIMENT_DIR))

from prepare_acdc_inputs import read_gre, read_scan, check_reference_sequence, check_physical_geometry
from read_acdc_sequence import read_sequence
from read_acdc_b0 import read_b0_coil_images, b0_from_coil_images
from read_acdc_calibration import read_calibration
from resample_acdc_maps import resample_to_grid
from run_acdc_experiment import check_calibration, current_windows
from export_acdc_waveform import export_waveform, quantize_currents

INPUT_DIR = PROJECT_ROOT / "acquisition_005"
FILES = {
    "reference": INPUT_DIR / "gre_reference.dat",
    "b0": INPUT_DIR / "b0_map.dat",
    "scan": INPUT_DIR / "scan.dat",
    "calibration": INPUT_DIR / "ACDC_3T.npz",
    "scan_sequence": INPUT_DIR / "sequences/scan.seq",
    "reference_sequence": INPUT_DIR / "sequences/reference_gre.seq",
    "b0_sequence": INPUT_DIR / "sequences/b0_map.seq",
}
SESSION = datetime.now().strftime("notebook_%Y%m%d_%H%M%S")
OUTPUT_DIR = EXPERIMENT_DIR / "acdc_output" / SESSION
# Python/MATLAB working files and worker storage share this scratch folder.
SCRATCH_DIR = Path("/scratch/mv1289")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
_workspace = tempfile.TemporaryDirectory(prefix="acdc_notebook_", dir=SCRATCH_DIR)
WORK_DIR = Path(_workspace.name)
print("Input folder:", INPUT_DIR)
print("CSV destination:", OUTPUT_DIR / "waveform.csv")
print("Temporary MATLAB workspace:", WORK_DIR)

## 2. Experiment parameters

These values are explicit choices for this dataset. `b0_sign` is applied **once** after
phase-based field estimation. The receive-coil phase convention and the absolute frequency
branch need independent experimental verification. Connected B0 components may have
unknown relative multiples of `1 / echo_spacing`.

The Bay5 registration and source hash below retain your previous confirmation.
The numerical current bounds are the values you entered. The final cell only places currents in
CSV columns and timestamps; it never connects to the controller.

In [ ]:
# Reconstruction and physical registration
REFERENCE_THRESHOLD = 0.2
B0_SIGN = 1
GEOMETRY_CONFIRMED = True
B0_SIGN_CONFIRMED = True
T2STAR_S = np.inf  # No measured T2* map: retain the existing no-decay assumption.
B0_PARAMETERS = {
    "magnitude_fraction": 0.05,
    "max_phase_residual_rad": 0.5,
    "min_component_pixels": 10,
    "frequency_offset_hz": 0.0,
}
CALIBRATION_PARAMETERS = {
    "units": "Hz/A",
    "registration_confirmed": True,
    "patient_to_source_m": np.eye(4).tolist(),
    "source_sha256": "bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note": "Bay5 coils and alignment confirmed by the user on 2026-10-03.",
}

# MATLAB simulation and optimization
MATLAB_EXECUTABLE = Path("/usr/pubsw/packages/matlab/26.1/bin/matlab")  # Replace on another computer.
RUN_FORWARD_MODEL = True
RUN_OPTIMIZATION = True
FIELD_LIMIT_HZ = 500.0
MAX_CURRENT_A = 1.0
TOTAL_CURRENT_LIMIT_A = 30.0
MAX_ITERATIONS = 500
WORKERS = 1
THREADS_PER_WORKER = 1
DEVICE = "gpu"

# CSV column order and numerical current limits.
WAVEFORM_PARAMETERS = {
    "channel_map": [i for i in range(1, 33)],  # Enter 32 distinct physical output numbers in 1..64, in calibration order.
    "channel_polarity": [1] * 32,
    "max_current_A": MAX_CURRENT_A,
    "total_current_limit_A": TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s": None,  # Optional positive minimum spacing between CSV rows.
}
EXPORT_WAVEFORM = True

In [ ]:
# Display diagnostics in the notebook without writing additional output files.
def show_figure(fig):
    plt.show()
    plt.close(fig)


def show_table(rows, headings=("Parameter", "Value")):
    lines = ["| " + " | ".join(headings) + " |", "| " + " | ".join(["---"] * len(headings)) + " |"]
    lines += ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def image_panel(ax, values, title, cmap="gray", **kwargs):
    artist = ax.imshow(values, cmap=cmap, interpolation="nearest", **kwargs)
    ax.set_title(title)
    ax.set_xlabel("Array column")
    ax.set_ylabel("Array row")
    return artist


for name, path in FILES.items():
    assert path.is_file(), f"Missing {name}: {path}"
assert not isinstance(B0_SIGN, bool) and B0_SIGN in (-1, 1), "B0_SIGN must be +1 or -1."
assert 0 <= REFERENCE_THRESHOLD < 1, "REFERENCE_THRESHOLD must be in [0, 1)."
assert np.isscalar(T2STAR_S) and T2STAR_S > 0, "T2STAR_S must be positive; infinity is allowed."
show_table([(name, path.name) for name, path in FILES.items()])

## 3. Sequence trajectory and acquisition timing

The trajectory is derived from the actual RF/ADC/gradient events. The `Matrix`, `FOV`,
`TE` and `TR` definitions are checked against these events. Cartesian order is used
for reconstruction, while `line_index` preserves chronological current order.
For spoiled GRE every line starts a new transverse phase history; EPI retains one RF history.

In [ ]:
# Read the scan and reference sequences independently.
acquisition, timing = read_sequence(FILES["scan_sequence"])
reference_acquisition, reference_timing = read_sequence(FILES["reference_sequence"])
show_table([
    ("Scan kind", timing["sequence_kind"]),
    ("Scan matrix [Nx, Ny]", timing["matrix_size"]),
    ("Scan TE / TR [ms]", np.array([acquisition["TE_s"], acquisition["TR_s"]]) * 1000),
    ("ADC dwell [us]", acquisition["dwell_s"] * 1e6),
    ("Reference matrix [Nx, Ny]", reference_timing["matrix_size"]),
    ("Reference TE / TR [ms]", np.array([reference_acquisition["TE_s"], reference_acquisition["TR_s"]]) * 1000),
])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
plot = axes[0].scatter(acquisition["kx_mInv"], acquisition["ky_mInv"],
                       c=acquisition["time_s"] * 1000, s=6, cmap="viridis")
axes[0].set(xlabel="kx [cycles/m]", ylabel="ky [cycles/m]", title="ADC trajectory, colored by time")
fig.colorbar(plot, ax=axes[0], label="Time from first RF center [ms]")
plot = image_panel(axes[1], acquisition["line_index"], "Chronological line index", cmap="viridis")
fig.colorbar(plot, ax=axes[1])
starts, ends = current_windows(timing)
for line, (start, end) in enumerate(zip(starts, ends)):
    axes[2].plot([start * 1000, end * 1000], [line + 1, line + 1], color="tab:blue")
for center in np.unique(acquisition["rf_time_s"]):
    axes[2].axvline(center * 1000, color="tab:red", alpha=0.3)
axes[2].set(xlabel="Time from first RF [ms]", ylabel="Current state", title="Current windows and RF centers")
show_figure(fig)

## 4. Reconstruct the reference and verify its physical coordinates

The reference is a **mono-echo Cartesian 2D GRE/FLASH Twix** with one RF excitation
per acquired line. Its own `reference_gre.seq` supplies the expected matrix, FOV,
TE, TR, ADC dwell, readout polarity and acquisition order. These are checked
against the raw data before defining the ROI; no image size is entered manually.

Reorder the acquired k-space lines, apply a centered inverse Fourier transform to
each receive coil, then combine magnitudes with root-sum-of-squares (RSS).
The array axes are `[phase row, readout column]`. Physical coordinates use
`X = readout` and `Y = phase`, in metres. The displayed pixel coordinates also
allow you to inspect the reference geometry before resampling the measured maps.

The RSS magnitude is an effective transverse-amplitude proxy, not a proton-density
or T1 map. The next cell defines the ROI from this reconstructed reference.


In [ ]:
# Reconstruct the raw reference and validate it against its own GRE sequence.
gre, reference_diagnostics = read_gre(
    FILES["reference"], reference_acquisition, return_intermediates=True)
check_reference_sequence(gre, reference_acquisition)
reference_magnitude = np.asarray(gre["magnitude"], float)
reference_metadata = gre["twix_metadata"]
X_m, Y_m = np.asarray(gre["X_m"]), np.asarray(gre["Y_m"])
show_table([
    ("Reference matrix [phase rows, readout columns]", reference_magnitude.shape),
    ("Receive coils", reference_metadata["receiver_count"]),
    ("TE / TR [ms]", np.array([reference_metadata["TE_s"], reference_metadata["TR_s"]]) * 1000),
    ("ADC dwell [us]", reference_metadata["dwell_s"] * 1e6),
    ("Readout / phase FOV [mm]", np.asarray(reference_metadata["FOV_m"]) * 1000),
    ("Slice center [patient mm]", reference_metadata["slice_position_mm"]),
])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(reference_diagnostics["kspace"], axis=1)), "Reference: log k-space RSS")
image_panel(axes[1], np.abs(reference_diagnostics["coil_images"][:, 0, :]), "Reference: receiver 1 magnitude")
image_panel(axes[2], reference_magnitude, "Reference: combined RSS magnitude")
show_figure(fig)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
plot = image_panel(axes[0], np.angle(reference_diagnostics["coil_images"][:, 0, :]),
                   "Reference: receiver 1 phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
fig.colorbar(plot, ax=axes[0])
axes[1].scatter(X_m * 1000, Y_m * 1000, c=reference_magnitude, s=8, cmap="gray")
axes[1].set(xlabel="X: readout [mm]", ylabel="Y: phase [mm]", title="Physical pixel coordinates", aspect="equal")
show_figure(fig)


### 4b. Define and inspect the ROI

The default selects all reference pixels above `REFERENCE_THRESHOLD * maximum_magnitude`.
The threshold is set in step 2. Optional X/Y limits select a rectangle in physical
millimetres; keep both limits at `None` to select the whole thresholded object.

Change these limits and rerun this cell and the following cells. Green pixels in the
overlay are the requested ROI. The binary mask and excluded object pixels show exactly
what was selected. The computed mask stays in memory and is included in the internal MATLAB input.
No mask is read from the input folder. B0 coverage is checked separately in step 8;
unreliable field measurements never shrink the requested ROI automatically.

In [ ]:
# Optional physical rectangle limits; None means no restriction on that axis.
ROI_X_LIMITS_MM = None  # Example: (-40.0, 40.0), X = readout.
ROI_Y_LIMITS_MM = None  # Example: (-40.0, 40.0), Y = phase.

threshold_support = reference_magnitude > REFERENCE_THRESHOLD * reference_magnitude.max()
support = threshold_support.copy()
for axis_name, limits, coordinates in (
    ("X", ROI_X_LIMITS_MM, X_m), ("Y", ROI_Y_LIMITS_MM, Y_m)
):
    if limits is not None:
        bounds = np.asarray(limits, dtype=float)
        assert bounds.shape == (2,) and np.isfinite(bounds).all() and bounds[0] <= bounds[1], f"{axis_name} limits need two finite increasing values in mm."
        support &= (coordinates * 1000 >= bounds[0]) & (coordinates * 1000 <= bounds[1])

support_before_validity_filter = support.copy()
excluded_object_pixels = threshold_support & ~support
excluded_signal_fraction = reference_magnitude[~support].sum() / reference_magnitude.sum()
fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout="constrained")
image_panel(axes[0, 0], reference_magnitude, "GRE reference magnitude")
image_panel(axes[0, 1], support, f"Computed ROI: {int(support.sum())} pixels", vmin=0, vmax=1)
image_panel(axes[1, 0], reference_magnitude, "GRE overlay: green = selected ROI")
axes[1, 0].imshow(np.ma.masked_where(~support, support), cmap="Greens", vmin=0, vmax=1, alpha=0.45, interpolation="nearest")
if support.any() and not support.all():
    axes[1, 0].contour(support, levels=[0.5], colors=["lime"], linewidths=0.8)
image_panel(axes[1, 1], excluded_object_pixels, "Thresholded object pixels outside ROI", cmap="Reds", vmin=0, vmax=1)
show_figure(fig)
show_table([
    ("Reference magnitude threshold", REFERENCE_THRESHOLD),
    ("X / Y limits [mm]", f"{ROI_X_LIMITS_MM} / {ROI_Y_LIMITS_MM}"),
    ("Thresholded object pixels", int(threshold_support.sum())),
    ("Selected ROI pixels", int(support.sum())),
    ("Reference magnitude excluded", f"{excluded_signal_fraction:.1%}"),
])
assert support.any(), "The computed ROI is empty: inspect the threshold and physical limits above."
ROI_PARAMETERS = {
    "method": "reference magnitude threshold with optional physical X/Y limits",
    "reference_threshold": REFERENCE_THRESHOLD,
    "x_limits_mm": [] if ROI_X_LIMITS_MM is None else list(ROI_X_LIMITS_MM),
    "y_limits_mm": [] if ROI_Y_LIMITS_MM is None else list(ROI_Y_LIMITS_MM),
}
print("ROI defined in memory; inspect the selection figure before continuing.")

## 5. Scan k-space, navigator correction and RSS reconstruction

The raw scan is reconstructed independently of the reference. For this EPI, three
navigator echoes estimate an affine odd/even phase error per receiver. Negative readouts
are reversed first; RSS then combines the receiver magnitudes. These intermediate arrays
come from the same reconstruction used by the folder pipeline.

In [ ]:
# Inspect reconstruction before and after navigator correction.
scan_image, scan_metadata, scan_diagnostics = read_scan(FILES["scan"], acquisition, return_intermediates=True)
check_physical_geometry(gre, scan_metadata, acquisition, support, X_m, Y_m, dicom_input=False)
for key in ("TE_s", "TR_s", "dwell_s"):
    tolerance = 1e-9 if key == "dwell_s" else 1e-6
    assert abs(scan_metadata[key] - acquisition[key]) <= tolerance, f"Scan header disagrees on {key}."
assert scan_metadata["navigator_count"] == timing["navigator_count"], "Navigator count mismatch."
assert np.array_equal(scan_metadata["polarity"], acquisition["polarity"]), "Readout polarity mismatch."
assert scan_metadata["acquired_phase_rows"] == np.argsort(acquisition["line_index"][:, 0]).tolist(), "Acquisition order mismatch."
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(scan_diagnostics["kspace"], axis=1)), "Raw reordered k-space: log RSS")
image_panel(axes[1], np.log1p(np.linalg.norm(scan_diagnostics["corrected_kspace"], axis=1)), "Corrected k-space: log RSS")
image_panel(axes[2], scan_diagnostics["uncorrected_image"], "RSS before navigator correction")
image_panel(axes[3], scan_image, "RSS after navigator correction")
show_figure(fig)
if scan_metadata["navigator_count"]:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    image_panel(axes[0], np.angle(scan_diagnostics["navigator_hybrid"][:, 0]), "Navigator phases, receiver 1", cmap="twilight", vmin=-np.pi, vmax=np.pi)
    plot = image_panel(axes[1], scan_diagnostics["navigator_phase_correction_rad"], "Fitted phase correction [rad]", cmap="RdBu_r")
    fig.colorbar(plot, ax=axes[1])
    show_figure(fig)
print("Scan matrix:", scan_image.shape, "; receiver count:", scan_metadata["receiver_count"])

## 6. Multi-echo B0: complex images before field estimation

B0 requires phase information from at least two equally spaced monopolar echoes.
First inspect each echo's k-space, RSS magnitude and one receiver's phase. This cell
runs before masking or phase unwrapping, so acquisition/reconstruction problems remain visible.

In [ ]:
# Read complex echoes without yet estimating a field.
b0_raw = read_b0_coil_images(FILES["b0"], FILES["b0_sequence"])
coils = b0_raw["coil_images"]  # [echo, receiver, phase row, readout column]
te_s = np.asarray(b0_raw["metadata"]["sequence_info"]["TE_s"])
assert abs(b0_raw["metadata"]["slice_position_mm"][2] - gre["position_mm"][2]) <= 1e-3, "B0 and reference slice planes differ."
assert np.isclose(abs(np.dot(b0_raw["metadata"]["slice_quaternion_wxyz"], scan_metadata["slice_quaternion_wxyz"])), 1, atol=1e-8), "B0 and scan orientations differ."
show_table([("B0 matrix", coils.shape[2:]), ("Echoes", coils.shape[0]),
            ("Receive coils", coils.shape[1]), ("Echo times [ms]", te_s * 1000)])
ne = coils.shape[0]
fig, axes = plt.subplots(3, ne, figsize=(5 * ne, 11), layout="constrained", squeeze=False)
for echo in range(ne):
    image_panel(axes[0, echo], np.log1p(np.linalg.norm(b0_raw["kspace"][echo], axis=0)), f"Echo {echo + 1}: log k-space RSS")
    image_panel(axes[1, echo], np.linalg.norm(coils[echo], axis=0), f"Echo {echo + 1}: magnitude RSS")
    image_panel(axes[2, echo], np.angle(coils[echo, 0]), f"Echo {echo + 1}: receiver 1 phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
show_figure(fig)

## 7. Phase differences, reliability masks and measured field

For each receiver, `echo[n+1] * conj(echo[n])` cancels its static receive phase. The
products are summed across receivers and echo pairs. The field convention is
`signal(TE) = amplitude * exp(-2π i B0 TE)`, followed by the explicit scanner sign choice.

Reliability requires sufficient magnitude, consistent adjacent-echo phase increments,
and components large enough for spatial phase unwrapping. No B0 value is extrapolated.
Each component is anchored at its strongest pixel in the principal frequency band;
this assumption does not measure the absolute frequency branch.

In [ ]:
# Show phase increments before spatial unwrapping.
echo_products = np.sum(coils[1:] * coils[:-1].conj(), axis=1)
wrapped_increment = np.angle(echo_products.sum(axis=0))
fig, axes = plt.subplots(1, ne, figsize=(5 * ne, 4), layout="constrained", squeeze=False)
for pair in range(ne - 1):
    image_panel(axes[0, pair], np.angle(echo_products[pair]), f"Echo pair {pair + 1}: wrapped increment", cmap="twilight", vmin=-np.pi, vmax=np.pi)
image_panel(axes[0, -1], wrapped_increment, "Combined wrapped increment [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
show_figure(fig)

In [ ]:
# Estimate the field with the same tested reconstruction as the pipeline.
b0_result = b0_from_coil_images(coils, te_s, return_intermediates=True, **B0_PARAMETERS)
b0_result["b0Hz"] *= B0_SIGN
b0_result["X_m"], b0_result["Y_m"] = b0_raw["X_m"], b0_raw["Y_m"]
b0_details = b0_result["intermediates"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout="constrained")
image_panel(axes[0, 0], b0_details["magnitude_mask"], "Magnitude threshold passed")
image_panel(axes[0, 1], b0_details["phase_residual_mask"], "Echo consistency passed")
image_panel(axes[0, 2], b0_result["mask"], "Final reliability mask")
plot = image_panel(axes[1, 0], b0_result["phase_residual_rad"], "Echo increment disagreement [rad]", cmap="magma", vmin=0, vmax=np.pi)
fig.colorbar(plot, ax=axes[1, 0])
plot = image_panel(axes[1, 1], b0_details["component_labels"], "Connected field components", cmap="tab20")
plot = image_panel(axes[1, 2], b0_result["b0Hz"], "Measured B0 [Hz], masked", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[1, 2])
show_figure(fig)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
image_panel(axes[0], b0_details["quality_before_component_filter"], "Magnitude and phase checks passed")
image_panel(axes[1], b0_result["mask"], "Large reliable components retained")
image_panel(axes[2], b0_details["quality_before_component_filter"] & ~b0_result["mask"], "Small components removed")
show_figure(fig)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.hist(b0_result["b0Hz"][b0_result["mask"]], bins=60)
ax.set(xlabel="Measured B0 [Hz]", ylabel="Reliable pixels", title="Masked field distribution")
show_figure(fig)
show_table([("Reliable native-grid pixels", int(b0_result["mask"].sum())),
            ("Frequency band at component anchors [Hz]", b0_result["metadata"]["principal_frequency_band_hz"]),
            ("B0 sign applied", B0_SIGN)])

## 8. Resample measured maps onto the reference grid

The B0 grid and reference grid are independent. Linear interpolation uses their physical
coordinates and accepts a target pixel only when every contributing source measurement is
valid. The full reference support and the selected ROI are both compared against coverage.
The requested ROI is kept unchanged. Missing B0 measurements are displayed explicitly
and block simulation until this coverage problem is resolved.

In [ ]:
# Interpolate without filling holes or extending the measured field.
b0_on_reference, b0_valid = resample_to_grid(
    b0_result["b0Hz"], b0_result["X_m"], b0_result["Y_m"], b0_result["mask"], X_m, Y_m)
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], threshold_support, "Full reference support")
image_panel(axes[1], b0_valid, "Valid B0 on reference grid")
image_panel(axes[2], support, "Selected simulation support")
plot = image_panel(axes[3], np.where(b0_valid, b0_on_reference, np.nan), "Measured B0 on reference [Hz]", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[3])
show_figure(fig)
missing_b0_pixels = support & ~b0_valid
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image_panel(ax, reference_magnitude, "Requested ROI: red = missing reliable B0")
ax.imshow(np.ma.masked_where(~missing_b0_pixels, missing_b0_pixels), cmap="Reds", vmin=0, vmax=1, alpha=0.65, interpolation="nearest")
if support.any() and not support.all():
    ax.contour(support, levels=[0.5], colors=["lime"], linewidths=0.8)
show_figure(fig)
show_table([("Full support covered by B0", f"{int((threshold_support & b0_valid).sum())}/{int(threshold_support.sum())}"),
            ("Selected ROI covered by B0", f"{int((support & b0_valid).sum())}/{int(support.sum())}"),
            ("Selected ROI pixels missing reliable B0", int(missing_b0_pixels.sum())),
            ("Reference signal excluded by ROI", f"{excluded_signal_fraction:.1%}")])
assert GEOMETRY_CONFIRMED and B0_SIGN_CONFIRMED, "Confirm the physical geometry and scanner sign in the parameter cell."
#assert not missing_b0_pixels.any(), f"Reliable B0 is missing for {int(missing_b0_pixels.sum())}/{int(support.sum())} requested ROI pixels. The ROI has not been reduced; inspect the coverage figures before simulation."

## 9. ACDC response maps and MATLAB input

The original NPZ contains a 3D response volume in Hz/A. The confirmed patient-to-source
transform samples that volume at the reference slice. The existing interpolation code
preserves channel order and leaves unknown responses as NaN. Show all 32 responses before
constructing the exact experiment struct consumed by the MATLAB functions.

In [ ]:
# Reuse the measured calibration import on this reference grid.
geometry = {key: gre[key] for key in ("X_m", "Y_m", "position_mm", "spacing_mm", "orientation", "slice_thickness_mm")}
calibration = read_calibration(FILES["calibration"], {"geometry": geometry}, CALIBRATION_PARAMETERS)
coil_maps = calibration["acdc_maps"]
calibration_valid = calibration["spatial_valid_mask"]
fig, axes = plt.subplots(4, 8, figsize=(19, 10), layout="constrained")
scale = np.nanpercentile(np.abs(coil_maps), 99)
for channel, ax in enumerate(axes.ravel()):
    ax.imshow(coil_maps[..., channel], cmap="RdBu_r", vmin=-scale, vmax=scale, interpolation="nearest")
    ax.set_title(f"Channel {channel + 1}")
    ax.set_axis_off()
fig.suptitle(f"Measured ACDC responses on reference grid [Hz/A], common scale ±{scale:.1f}")
show_figure(fig)
print("Calibration shape:", coil_maps.shape, "; units:", calibration["units"])

In [ ]:
TRIM_TO_COMMON_VALID_SUPPORT = True
assert support_before_validity_filter.shape == calibration_valid.shape == b0_valid.shape, "ROI, ACDC, and B0 masks must share the reference grid."
valid_acdc = calibration_valid & np.isfinite(coil_maps).all(axis=-1)
valid_b0 = b0_valid & np.isfinite(b0_on_reference)
common_valid = valid_acdc & valid_b0
acdc_invalid_pixels = support_before_validity_filter & ~valid_acdc
b0_invalid_pixels = support_before_validity_filter & ~valid_b0
invalid_for_either = acdc_invalid_pixels | b0_invalid_pixels
invalid_for_both = acdc_invalid_pixels & b0_invalid_pixels
removed_pixels = invalid_for_either.copy() if TRIM_TO_COMMON_VALID_SUPPORT else np.zeros_like(support_before_validity_filter, dtype=bool)
support = support_before_validity_filter & common_valid if TRIM_TO_COMMON_VALID_SUPPORT else support_before_validity_filter.copy()
excluded_object_pixels = threshold_support & ~support
excluded_signal_fraction = reference_magnitude[~support].sum() / reference_magnitude.sum()
ROI_PARAMETERS.update({
    "common_validity_filter_enabled": bool(TRIM_TO_COMMON_VALID_SUPPORT),
    "selected_pixels_invalid_for_acdc_calibration": int(acdc_invalid_pixels.sum()),
    "selected_pixels_invalid_for_b0": int(b0_invalid_pixels.sum()),
    "selected_pixels_invalid_for_both": int(invalid_for_both.sum()),
    "pixels_removed_for_invalid_acdc_or_b0": int(removed_pixels.sum()),
    "selected_pixels_after_common_validity_filter": int(support.sum()),
})
fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="constrained")
image_panel(axes[0], support_before_validity_filter, f"Original support: {int(support_before_validity_filter.sum())} pixels", vmin=0, vmax=1)
image_panel(axes[1], valid_acdc, "Valid ACDC responses", vmin=0, vmax=1)
image_panel(axes[2], valid_b0, "Finite, valid B0", vmin=0, vmax=1)
image_panel(axes[3], support, f"Common-valid support: {int(support.sum())} pixels", vmin=0, vmax=1)
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image_panel(ax, reference_magnitude, f"Pixels removed for invalid ACDC or B0: {int(removed_pixels.sum())}")
removed_rows, removed_columns = np.where(removed_pixels)
ax.scatter(removed_columns, removed_rows, marker="s", s=28, c="red")
show_figure(fig)
show_table([("Common-validity filter enabled", TRIM_TO_COMMON_VALID_SUPPORT),
            ("Pixels invalid for ACDC", int(acdc_invalid_pixels.sum())),
            ("Pixels invalid for B0", int(b0_invalid_pixels.sum())),
            ("Pixels invalid for both", int(invalid_for_both.sum())),
            ("Pixels removed (unique total)", int(removed_pixels.sum())),
            ("Selected pixels remaining", int(support.sum())),
            ("Reference signal excluded", f"{excluded_signal_fraction:.1%}")])
assert support.any(), "ACDC/B0 validity filtering removed the entire selected support."
assert np.all(common_valid[support]), "Selected support contains an invalid ACDC or B0 pixel; enable TRIM_TO_COMMON_VALID_SUPPORT."

In [ ]:
# Build the existing MATLAB struct transparently; no new forward model is implemented in Python.
scale = reference_magnitude[support].max()
m0_proxy = np.where(support, reference_magnitude / scale, 0.0)
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in FILES.items()}
experiment = {
    "images": {"gre_reference": reference_magnitude, "scan": scan_image, "perturbed": scan_image},
    "field": {"b0Hz": np.where(support, b0_on_reference, 0.0),
              "measured_b0Hz": np.where(b0_valid, b0_on_reference, np.nan),
              "mask": b0_valid, "simulation_mask": support},
    "geometry": geometry, "acquisition": acquisition, "m0_proxy": m0_proxy,
    "relaxation": {"T2star_s": T2STAR_S},
    "assumptions": np.array([
        "GRE magnitude is a transverse-amplitude proxy; reference/scan contrast is not calibrated.",
        "Only the explicitly chosen ROI is simulated; every ROI field value is measured.",
        "Uniform receive sensitivity; instantaneous current transitions; ideal GRE spoiling.",
        "Component frequency branches and scanner B0 sign need independent verification.",
    ], dtype=object),
    "provenance": {"source_sha256": source_hashes, "B0_sign": B0_SIGN,
                   "GRE_amplitude_normalization": float(scale), "roi_parameters": ROI_PARAMETERS,
                   "selected_roi_sha256": hashlib.sha256(np.ascontiguousarray(support, dtype=np.uint8).tobytes()).hexdigest()},
    "optimization_ready": True,
}
check_calibration(calibration, experiment)
INPUT_MAT = WORK_DIR / "experiment_input.mat"
CALIBRATION_MAT = WORK_DIR / "calibration.mat"
savemat(INPUT_MAT, {"experiment": experiment}, do_compression=True, long_field_names=True)
savemat(CALIBRATION_MAT, calibration, do_compression=True, long_field_names=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
image_panel(axes[0], m0_proxy, "Normalized MATLAB object amplitude")
plot = image_panel(axes[1], np.where(support, b0_on_reference, np.nan), "Measured field used by MATLAB [Hz]", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[1])
show_figure(fig)
print("Internal MATLAB inputs:", INPUT_MAT, CALIBRATION_MAT)

## 10. Run the existing MATLAB forward model

The notebook writes a MAT request and calls `run_acdc_notebook_stage`. MATLAB evaluates
`forward_model_experiment` twice on exactly the selected object: once with B0 = 0 and once
with measured B0. Comparing those twins tests the simulated effect of the field.
The measured scan is displayed separately because reference/scan contrast and receive
sensitivity are not calibrated by this model.

In [ ]:
# The bridge only orchestrates the existing MATLAB functions.
request = {
    "input_file": str(INPUT_MAT), "calibration_file": str(CALIBRATION_MAT),
    "output_dir": str(WORK_DIR / "matlab"), "max_current_A": MAX_CURRENT_A,
    "total_current_limit_A": TOTAL_CURRENT_LIMIT_A, "field_limit_Hz": FIELD_LIMIT_HZ,
    "max_iterations": float(MAX_ITERATIONS), "workers": float(WORKERS),
    "threads_per_worker": float(THREADS_PER_WORKER), "device": DEVICE, "scratch_dir": str(SCRATCH_DIR),
}
REQUEST_MAT = WORK_DIR / "matlab_request.mat"
savemat(REQUEST_MAT, {"request": request}, long_field_names=True)


def run_matlab_stage(stage):
    assert MATLAB_EXECUTABLE.is_file(), f"MATLAB executable not found: {MATLAB_EXECUTABLE}"
    quote = lambda path: str(path).replace("'", "''")
    command = f"addpath('{quote(EXPERIMENT_DIR)}'); run_acdc_notebook_stage('{quote(REQUEST_MAT)}','{stage}');"
    log_path = WORK_DIR / f"matlab_{stage}.log"
    print(f"Running MATLAB {stage}; full log: {log_path}", flush=True)
    environment = os.environ.copy()
    environment["TMPDIR"] = str(SCRATCH_DIR)
    with log_path.open("w") as log:
        completed = subprocess.run([str(MATLAB_EXECUTABLE), "-batch", command], stdout=log, stderr=subprocess.STDOUT, env=environment)
    print("\n".join(log_path.read_text().splitlines()[-12:]))
    assert completed.returncode == 0, f"MATLAB failed; inspect {log_path}."
    result_name = "forward.mat" if stage == "forward" else "optimization_stage.mat"
    result = loadmat(WORK_DIR / "matlab" / result_name, simplify_cells=True)
    assert result["status"]["state"] == "complete", result["status"]["message"]
    return result


forward = run_matlab_stage("forward") if RUN_FORWARD_MODEL else None
if forward is None:
    print("Forward stage disabled explicitly in the parameter cell.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
    twin_scale = max(np.abs(forward["ideal_image"]).max(), np.abs(forward["measured_image"]).max())
    image_panel(axes[0], np.abs(forward["ideal_image"]), "MATLAB: ideal zero-B0 magnitude", vmin=0, vmax=twin_scale)
    image_panel(axes[1], np.abs(forward["measured_image"]), "MATLAB: measured-B0 prediction", vmin=0, vmax=twin_scale)
    image_panel(axes[2], np.abs(np.abs(forward["measured_image"]) - np.abs(forward["ideal_image"])), "Absolute magnitude difference", cmap="magma")
    image_panel(axes[3], scan_image, "Measured RSS scan: own scale")
    show_figure(fig)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    for ax, values, title in zip(axes,
                                 (forward["ideal_kdata"], forward["measured_kdata"], forward["measured_kdata"] - forward["ideal_kdata"]),
                                 ("Ideal k-space", "Measured-B0 k-space", "Complex k-space difference")):
        image_panel(ax, np.log1p(np.abs(values)), title + ": log(1 + magnitude)", cmap="magma")
    show_figure(fig)
    print("Initial magnitude relative MSE:", forward["relative_mse"])

## 11. MATLAB optimization and convergence

The existing optimizer minimizes magnitude image error to the same zero-B0 twin. It bounds
currents and the **generated ACDC field**, rather than bounding the residual B0 + ACDC field.
The numerical optimization can be inspected independently of wiring/trigger parameters.
A solver result is eligible for export only when convergence and all constraints pass.

In [ ]:
THREADS_PER_WORKER = 6
request["threads_per_worker"] = float(THREADS_PER_WORKER)
savemat(REQUEST_MAT, {"request": request}, long_field_names=True)

In [ ]:
# Run MATLAB optimization with the explicit numerical limits above.
optimization_stage = run_matlab_stage("optimize") if RUN_OPTIMIZATION else None
solution = None
if optimization_stage is not None:
    files = list((WORK_DIR / "matlab/optimization").glob("solution_*Hz.mat"))
    assert len(files) == 1, "Expected one solver result for the selected field limit."
    solution = loadmat(files[0], simplify_cells=True)
    saved_starts = np.asarray(solution["line_start_s"]).ravel()
    saved_ends = np.asarray(solution.get("line_end_s", [])).ravel()
    solution_checks = [
        ("Registered calibration used", solution.get("calibration_mode") == "registered" and bool(solution.get("registration_confirmed"))),
        ("Current dimensions match this sequence", np.shape(solution["currents_A"]) == (32, len(starts))),
        ("Start times match this sequence", saved_starts.shape == starts.shape and np.allclose(saved_starts, starts, atol=1e-12, rtol=0)),
        ("GRE end times match this sequence", timing["sequence_kind"] != "gre" or (saved_ends.shape == ends.shape and np.allclose(saved_ends, ends, atol=1e-12, rtol=0))),
        ("Saved field bound matches this run", float(solution["field_limit_Hz"]) == FIELD_LIMIT_HZ),
    ]
    show_table([(name, "PASS" if passed else "FAIL") for name, passed in solution_checks], ("Saved result check", "Result"))
    assert all(passed for _, passed in solution_checks), "Saved MATLAB results do not match these inputs; rerun from the parameter cell."
    info = solution["optimization_info"]
    show_table([(key, info[key]) for key in ("exitflag", "fval", "converged", "current_limits_met", "field_limits_met")])
    history = solution["history"]
    iterations = np.atleast_1d(history["iteration"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    for ax, key, title in zip(axes, ("fval", "firstorderopt", "constrviolation"),
                              ("Magnitude relative MSE", "First-order optimality", "Constraint violation")):
        values = np.atleast_1d(history[key])
        ax.plot(iterations, values, ".-")
        ax.set(xlabel="Iteration", ylabel=title, title=title)
    show_figure(fig)
else:
    print("Optimization disabled explicitly in the parameter cell.")

## 12. Corrected image, currents and generated fields

The corrected image and k-space below are the MATLAB outputs. Image errors refer to the
simulated ideal object on the selected ROI. They do not demonstrate a measured hardware
improvement. The current and field plots verify chronological states and constraints.
The second row of the dynamic field figure shows instantaneous `B0 + ACDC`; the
100 Hz constraint applies to the generated ACDC field in the first row. The optimizer
minimizes the image magnitude error rather than the instantaneous residual field.

In [ ]:
# Inspect solver outputs even when convergence is not achieved.
if solution is not None:
    ideal = solution["reference"]
    corrected = solution["image"]
    baseline = loadmat(WORK_DIR / "matlab/optimization/baseline.mat", simplify_cells=True)
    before = baseline["initial_image"]
    scale = max(np.abs(ideal).max(), np.abs(before).max(), np.abs(corrected).max())
    fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
    for ax, values, title in zip(axes[:3], (ideal, before, corrected),
                                 ("Ideal zero-B0 twin", "Initial measured-B0 twin", "Optimized ACDC twin")):
        image_panel(ax, np.abs(values), title, vmin=0, vmax=scale)
    image_panel(axes[3], np.abs(np.abs(corrected) - np.abs(ideal)), "Final magnitude difference", cmap="magma")
    show_figure(fig)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    for ax, values, title in zip(axes,
                                 (baseline["initial_kdata"], solution["kdata"], solution["kdata"] - baseline["initial_kdata"]),
                                 ("Initial k-space", "Optimized k-space", "Correction-induced k-space change")):
        image_panel(ax, np.log1p(np.abs(values)), title + ": log(1 + magnitude)", cmap="magma")
    show_figure(fig)
    currents = solution["currents_A"]
    generated_fields = solution["generated_field_Hz"]
    maximum_field = np.max(np.abs(generated_fields[support]), axis=0)
    total_current = np.abs(currents).sum(axis=0)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
    plot = image_panel(axes[0], currents, "Current per channel and acquired state [A]", cmap="RdBu_r")
    axes[0].set(xlabel="Chronological current state", ylabel="Calibration channel")
    fig.colorbar(plot, ax=axes[0])
    axes[1].plot(total_current)
    axes[1].axhline(TOTAL_CURRENT_LIMIT_A, color="red", linestyle="--", label="Total bound")
    axes[1].set(xlabel="Current state", ylabel="Sum |I| [A]", title="Total current per state")
    axes[1].legend()
    axes[2].plot(maximum_field)
    axes[2].axhline(FIELD_LIMIT_HZ, color="red", linestyle="--", label="Generated-field bound")
    axes[2].set(xlabel="Current state", ylabel="Max |ACDC field| [Hz]", title="Generated field on selected ROI")
    axes[2].legend()
    show_figure(fig)
    chosen_states = np.unique(np.linspace(0, currents.shape[1] - 1, 4).astype(int))
    residual_fields = b0_on_reference[..., None] + generated_fields
    residual_scale = np.nanmax(np.abs(np.r_[b0_on_reference[support], residual_fields[support].ravel()]))
    fig, axes = plt.subplots(2, len(chosen_states), figsize=(4 * len(chosen_states), 8), layout="constrained", squeeze=False)
    for column, state in enumerate(chosen_states):
        plot = image_panel(axes[0, column], np.where(support, generated_fields[..., state], np.nan),
                           f"ACDC field, state {state + 1} [Hz]", cmap="RdBu_r", vmin=-FIELD_LIMIT_HZ, vmax=FIELD_LIMIT_HZ)
        fig.colorbar(plot, ax=axes[0, column])
        plot = image_panel(axes[1, column], np.where(support, residual_fields[..., state], np.nan),
                           f"B0 + ACDC, state {state + 1} [Hz]", cmap="RdBu_r", vmin=-residual_scale, vmax=residual_scale)
        fig.colorbar(plot, ax=axes[1, column])
    show_figure(fig)
    current_ok = np.all(np.abs(currents) <= MAX_CURRENT_A + 1e-8) and np.all(total_current < TOTAL_CURRENT_LIMIT_A)
    field_ok = np.max(maximum_field) <= FIELD_LIMIT_HZ + 1e-8
    quantized_field_max = np.max(np.abs(coil_maps[support] @ quantize_currents(currents)))
    quantized_field_ok = quantized_field_max <= FIELD_LIMIT_HZ + 1e-8
    #solver_ok = bool(info["converged"]) and float(info["exitflag"]) > 0 and current_ok and field_ok and quantized_field_ok
    solver_ok = current_ok and field_ok and quantized_field_ok
    show_table([("Recomputed current constraints", current_ok), ("Recomputed field constraint", field_ok),
                ("Nominal quantized field maximum [Hz]", quantized_field_max), ("Solver eligible for export", solver_ok)])
else:
    solver_ok = False

## 13. Preview and export one waveform CSV

The CSV contains time in seconds followed by currents in amperes in physical output order.
It starts at time zero with the first current state and ends with a return to zero. For GRE, zero
states are inserted between acquired lines so the current is zero during RF excitations.
EPI retains its chronological imaging states.

Timestamps are shifted so that the first waveform state is at time zero. No trigger is sent.
`channel_map` and `channel_polarity` define the CSV columns; no hardware is configured.

The output folder receives **only `waveform.csv`**. No scripts, manifests or controller
commands are generated. The figure below previews the CSV values when export succeeds.

In [ ]:
# Generate data only. The FPGA is operated separately from this notebook.
csv_file = "waveform.csv"
if EXPORT_WAVEFORM and solver_ok:
    if timing["sequence_kind"] == "gre":
        csv_edges = np.column_stack((starts, ends)).ravel()
        csv_currents = np.zeros((currents.shape[0], 2 * currents.shape[1] - 1))
        csv_currents[:, ::2] = currents
        for start, end in zip(starts, ends):
            overlaps_rf = (np.asarray(timing["rf_start_s"]) < end - 1e-12) & (np.asarray(timing["rf_end_s"]) > start + 1e-12)
            assert not overlaps_rf.any(), "A current window overlaps a GRE RF pulse."
    else:
        csv_edges = np.r_[starts, ends[-1]]
        csv_currents = currents
    csv_edges -= csv_edges[0]
    from importlib import reload
    import export_acdc_waveform
    export_waveform = reload(export_acdc_waveform).export_waveform
    try:
        export_waveform(csv_currents, csv_edges, WAVEFORM_PARAMETERS, OUTPUT_DIR)
        csv_file = OUTPUT_DIR / "waveform.csv"
    except (ValueError, TypeError, FileExistsError) as error:
        print("CSV export blocked:", error)
else:
    print("CSV export disabled or numerical solution not eligible.")

if csv_file is not None:
    waveform = np.loadtxt(csv_file, delimiter=",")
    fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
    ax.step(waveform[:, 0] * 1000, waveform[:, 1:], where="post", alpha=0.6)
    ax.set(xlabel="Time from first waveform state [ms]", ylabel="Physical output current [A]", title="Exported waveform: final zero current")
    show_figure(fig)
    print("Only exported file:", csv_file)
print("Figures remain in this notebook. MATLAB working files are temporary.")

## Reading the result

1. Inspect the echo magnitudes, phase consistency and B0 components before trusting the field.
2. Compare the requested ROI with B0 coverage. The ROI is never reduced automatically.
3. Check the MATLAB images, solver history and current/field constraints.
4. Enter the CSV column mapping and desired time origin, then inspect the waveform preview.

Save the notebook to preserve its diagnostic figures. The delivered result is `waveform.csv`.
The MATLAB inputs, logs and results live in a temporary workspace for the current kernel.
The notebook does not upload a waveform, send a trigger or operate the FPGA.